<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking.**
My lane is CTR / Search Performance Optimization. This isn't classification —
there's no clean yes/no label for "should this page be fixed." It's not
clustering either, since I'm not grouping similar pages, I'm prioritizing them.
It's a **scoring** problem: assign each page an opportunity score so the content
team can rank pages and work down the list, starting with the highest-value fixes.

In [6]:
# No code needed for this section — task type is a framing decision
print("Task type: Scoring / Ranking")
print("Lane: CTR / Search Performance Optimization")

Task type: Scoring / Ranking
Lane: CTR / Search Performance Optimization


## 2. Target or proxy
*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target: an "opportunity score."**
There's no direct ground-truth label for "worth optimizing" — I can't observe
whether a title/metadata change would increase CTR without actually running
the change. So I build a proxy from observed data: the gap between a page's
actual CTR and the average CTR of other pages at a similar average search
position. A large positive gap (position is fine, but CTR is well below peers
at that position) signals an opportunity.

This is a proxy, not an observed outcome — it's inferred from a defined rule
applied to real numbers, not a confirmed "this fix worked" result.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

df_valid = df[df["avg_position"] > 0].copy()
df_valid["ctr_by_position_avg"] = df_valid.groupby("position_tier")["ctr"].transform("mean")
df_valid["opportunity_score"] = (df_valid["impressions_last_30d"] *
                            (df_valid["ctr_by_position_avg"] - df_valid["ctr"]).clip(lower=0))

print("Proxy target: opportunity_score")
print("Sample:")
print(df_valid[["content_id", "position_tier", "ctr", "ctr_by_position_avg", "opportunity_score"]].head())

Proxy target: opportunity_score
Sample:
             content_id position_tier   ctr  ctr_by_position_avg  \
0  content_304f48230142      striking  0.76             0.323239   
1  content_a1fb4e703a9e      page_3_5  0.05             0.222484   
2  content_9aa793d4d895      page_3_5  0.09             0.222484   
3  content_331d6c4de07b        page_1  0.49             0.652467   
4  content_d99b7a2d90ca      page_3_5  0.13             0.222484   

   opportunity_score  
0           0.000000  
1         431.382785  
2         315.577175  
3         589.103765  
4         389.450631  


## 3. Success metric
*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**
Of the top 50 pages the score ranks as highest-opportunity, what fraction
genuinely have CTR meaningfully below their position-tier peers? This matters
more than a generic error metric because the content team will only act on a
short list — what matters is whether the top of that list is worth their time.

In [8]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = (df_valid["ctr"] < df_valid["ctr_by_position_avg"]).astype(int).values
scores = df_valid["opportunity_score"].values

p_at_50 = precision_at_k(scores, labels, 50)
base_rate = labels.mean()

print(f"Precision@50: {p_at_50:.2f}")
print(f"Base rate: {base_rate:.2f}")
print(f"Lift over random: {p_at_50 - base_rate:.2f}")

Precision@50: 1.00
Base rate: 0.83
Lift over random: 0.17


## 4. The unit of analysis, as a real dataframe
*Load your lane's slice and show it: one row = one what?*

One row = one page (content_id), described by its last-30-day search
performance and its computed opportunity score.

In [9]:
unit_cols = ["content_id", "impressions_last_30d", "clicks_last_30d",
             "ctr", "avg_position", "position_tier", "opportunity_score"]

print("One row = one page (content_id)")
print(f"Total rows: {len(df_valid)}")
df_valid[unit_cols].sort_values("opportunity_score", ascending=False).head(10)

One row = one page (content_id)
Total rows: 28795


,content_id,impressions_last_30d,clicks_last_30d,ctr,avg_position,position_tier,opportunity_score
7678,content_8451fc6f034d,168958,36,0.03,2.3,top_3,462007.778405
3331,content_4a6607efcb46,122303,10,0.01,2.2,top_3,336877.914794
26844,content_8c19996aa890,89463,251,0.15,2.5,top_3,233896.844973
14090,content_44e481c8f55b,104458,623,0.65,1.4,top_3,220871.573781
21565,content_9532f197bbc8,109317,1176,0.87,2.0,top_3,207095.962876
21819,content_4c36c775b818,83723,548,0.41,2.3,top_3,197121.902428
3295,content_4fc39a2b8cf0,60914,343,0.69,2.6,top_3,126363.254713
18870,content_db5989a78dd3,238796,501,0.21,5.4,page_1,105659.245878
2346,content_11900bd7941a,40807,135,0.41,2.8,top_3,96078.180099
8578,content_23d452af4198,46561,243,0.77,2.4,top_3,92863.744991


## 5. Why ML beats a fixed rule here
*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "impressions > 1,000 and CTR < 5%") ignores context: expected
CTR varies a lot by position. A page at position 8 with 4% CTR may be performing
above expectation for that position, while a page at position 2 with the same
4% CTR is badly underperforming. A flat threshold can't tell these apart — it
would flag the wrong page and miss the real opportunity. Comparing each page
against its position-tier peers captures that context, which a single
if-statement cannot.

In [10]:
# Show why a flat rule fails — CTR varies significantly by position tier
tier_stats = df_valid.groupby("position_tier").agg(
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
    n=("ctr", "size")
).reset_index()

print("CTR varies by position tier — a flat threshold treats all tiers the same:")
print(tier_stats)

CTR varies by position tier — a flat threshold treats all tiers the same:
  position_tier  median_ctr  mean_ctr      n
0          deep        0.00  0.150212   1319
1        page_1        0.16  0.652467  11814
2      page_3_5        0.03  0.222484   7242
3      striking        0.11  0.323239   7304
4         top_3        0.00  2.764453   1116


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.